In [3]:
import numpy as np
import pandas as pd
import requests
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBClassifier
# Загрузка данных с Мосбиржи и подготовка признаков
def load_moex_data(ticker='SBER', start='2025-02-20', end='2028-12-31', board='TQBR'):
  url = f'https://iss.moex.com/iss/history/engines/stock/markets/shares/boards/{board}/securities/{ticker}.json'
  all_data = []
  start_idx = 0
  while True:
    params = {'from': start, 'till': end, 'start': start_idx}
    response = requests.get(url, params=params)
    data = response.json()
    history = data.get('history', {})
    columns = history.get('columns', [])
    rows = history.get('data', [])
    if not rows:
      break
    df_chunk = pd.DataFrame(rows, columns=columns)
    all_data.append(df_chunk)
    start_idx += len(rows)
    if len(rows) < 100:  # Мосбиржа отдает данные порциями
      break
  if not all_data:
    raise ValueError(
        'Не удалось получить данные с Мосбиржи. Проверьте параметры запроса.'
    )
  df = pd.concat(all_data, ignore_index=True)
  df = df[['TRADEDATE', 'CLOSE', 'VOLUME']].rename(
      columns={
          'TRADEDATE': 'Date',
          'CLOSE': 'Закрытие',
          'VOLUME': 'Объем',
      }
  )
  df['Date'] = pd.to_datetime(df['Date'])
  df.set_index('Date', inplace=True)
  df = df.astype(float)
  return df
# Определение функций
def Data(ticker='SBER', start='2025-02-20', end='2028-12-31'):
  df = load_moex_data(ticker, start, end)
  df['Доходность'] = df['Закрытие'].pct_change()
  df['SMA_10'] = df['Закрытие'].rolling(window=10).mean() / df['Закрытие']
  df['SMA_50'] = df['Закрытие'].rolling(window=50).mean() / df['Закрытие']
  df['Волатильность'] = df['Доходность'].rolling(window=10).std()
  delta = df['Закрытие'].diff()
  gain = delta.where(delta > 0, 0).rolling(window=14).mean()
  loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
  rs = gain / loss
  df['RSI'] = 100 - (100 / (1 + rs))
  df['Цель'] = (df['Закрытие'].shift(-1) > df['Закрытие']).astype(int)
  df.dropna(inplace=True)
  return df
# Создание модели
def Model():
  df = Data('SBER')
  features = ['Доходность', 'SMA_10', 'SMA_50', 'Волатильность', 'RSI', 'Объем']
  X = df[features]
  y = df['Цель']
  tscv = TimeSeriesSplit(n_splits=5)
  train_index, test_index = list(tscv.split(X))[-1]
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]
  model = XGBClassifier(
      n_estimators=200,
      max_depth=5,
      learning_rate=0.05,
      subsample=0.8,
      colsample_bytree=0.8,
      random_state=50,
  )
  model.fit(X_train, y_train)
  preds = model.predict(X_test)
  print(f'Точность: {accuracy_score(y_test, preds):.4f}\n')
  print('Отчет о классификации:')
  print(classification_report(y_test, preds))
  latest_features = X.iloc[[-1]]
  prob_up = model.predict_proba(latest_features)[0][1]
  print(f'Вероятность роста цены в следующий день: {prob_up * 100:.2f}%')
if __name__ == '__main__':
  Model()

Точность: 0.5000

Отчет о классификации:
              precision    recall  f1-score   support

           0       0.51      0.66      0.58        32
           1       0.48      0.33      0.39        30

    accuracy                           0.50        62
   macro avg       0.49      0.49      0.48        62
weighted avg       0.49      0.50      0.49        62

Вероятность роста цены в следующий день: 32.53%
